# Análise de Género nas Citações de Notícias Portuguesas

Pipeline para:
1. Extrair citações directas e indirectas dos artigos
2. Identificar o orador/a (Named Entity Recognition)
3. Inferir género a partir do primeiro nome
4. Agregar estatísticas por jornal, data, etc.

In [ ]:
import polars as pl
import re
import json
from pathlib import Path
from collections import Counter
import gc
import ctypes

def free_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

## 1. Carregar artigos

In [ ]:
DATA_DIR = Path("../data-scraping/data")

ALL_UTF8 = {col: pl.Utf8 for col in [
    "id", "url", "title", "subtitle", "text", "date", "author",
    "agency", "section", "source", "timestamp", "archive_url",
    "domain", "text_hash", "fetched_at", "extractor"
]}

jsonl_files = sorted(DATA_DIR.glob("*/articles.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files")

dfs = []
for f in jsonl_files:
    try:
        df = pl.read_ndjson(f, schema_overrides=ALL_UTF8).select(
            ["id", "text", "domain", "date"]
        )
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f.parent.name}: {e}")

articles = pl.concat(dfs)
del dfs
free_memory()

# Drop articles without text
articles = articles.filter(pl.col("text").is_not_null() & (pl.col("text").str.len_chars() > 100))
print(f"Loaded {len(articles):,} articles with text")

## 2. Extracção de citações

Estratégia:
- Citações directas: texto entre `«»`, `""`, `\u201c\u201d` seguido/precedido de verbo de atribuição
- Extrair o contexto em redor da citação para encontrar o orador

In [ ]:
# Verbos de atribuição comuns em jornalismo português
ATTRIBUTION_VERBS = [
    "disse", "afirmou", "declarou", "explicou", "referiu", "sublinhou",
    "acrescentou", "revelou", "garantiu", "admitiu", "considerou",
    "defendeu", "salientou", "alertou", "lamentou", "frisou",
    "recordou", "avançou", "anunciou", "confirmou", "sustentou",
    "notou", "precisou", "indicou", "destacou", "apontou",
    "ressalvou", "realçou", "assegurou", "reconheceu", "criticou",
    "elogiou", "questionou", "respondeu", "contou", "relatou",
    "escreveu", "comentou", "vincou", "reiterou", "insistiu",
    "pediu", "exigiu", "propôs", "sugeriu", "advertiu",
    "diz", "afirma", "declara", "explica", "refere", "sublinha",
    "acrescenta", "revela", "garante", "admite", "considera",
    "defende", "salienta", "alerta", "lamenta", "frisa",
    "recorda", "avança", "anuncia", "confirma", "sustenta",
    "nota", "precisa", "indica", "destaca", "aponta",
]

VERBS_PATTERN = "|".join(ATTRIBUTION_VERBS)

# Pattern: captures quote + surrounding context for speaker identification
# Group 1: context before quote (up to 100 chars)
# Group 2: the quote itself
# Group 3: context after quote (up to 100 chars)
QUOTE_PATTERNS = [
    # «quote», verb speaker  OR  speaker verb «quote»
    re.compile(
        r'(.{0,100})[«\u201c"](.*?)[»\u201d"](.{0,100})',
        re.DOTALL
    ),
]

# Pattern to find verb + speaker near quote
SPEAKER_AFTER_QUOTE = re.compile(
    rf'[»\u201d"]\s*,?\s*(?:{VERBS_PATTERN})\s+([A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+){{1,4}})',
    re.UNICODE
)

SPEAKER_BEFORE_QUOTE = re.compile(
    rf'([A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+){{1,4}})\s+(?:{VERBS_PATTERN})\s*:?\s*[«\u201c"]',
    re.UNICODE
)

# Simpler pattern: verb + name (for indirect speech)
# Use re.IGNORECASE for the lead words since they may be capitalised at sentence start
INDIRECT_SPEECH = re.compile(
    rf'(?:[Ss]egundo|[Dd]e acordo com|[Cc]onforme)\s+([A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+){{1,4}})',
    re.UNICODE
)

print(f"Compiled {len(ATTRIBUTION_VERBS)} attribution verbs")
print(f"Quote patterns ready")

In [ ]:
def extract_speakers(text: str) -> list[str]:
    """Extract speaker names from an article text using quote attribution patterns."""
    if not text:
        return []
    
    speakers = []
    
    # Direct quotes: speaker after quote ("...", disse Fulano)
    for m in SPEAKER_AFTER_QUOTE.finditer(text):
        speakers.append(m.group(1).strip())
    
    # Direct quotes: speaker before quote (Fulano disse: "...")
    for m in SPEAKER_BEFORE_QUOTE.finditer(text):
        speakers.append(m.group(1).strip())
    
    # Indirect speech (segundo Fulano, de acordo com Fulano)
    for m in INDIRECT_SPEECH.finditer(text):
        speakers.append(m.group(1).strip())
    
    return speakers

# Quick test
test_text = '«Estamos a trabalhar para resolver a situação», afirmou António Costa. Segundo Maria Silva, o problema está identificado.'
print(extract_speakers(test_text))

## 3. Processar artigos em batches

Extrair oradores de todos os artigos. Processamos em batches para não estourar a RAM.

In [ ]:
BATCH_SIZE = 100_000
QUOTES_FILE = Path("output/quotes_raw.jsonl")
QUOTES_FILE.parent.mkdir(exist_ok=True)

n_articles = len(articles)
total_quotes = 0

# Process in slices directly from Polars — never materialize all texts at once
with open(QUOTES_FILE, "w") as fout:
    for batch_start in range(0, n_articles, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_articles)
        batch = articles.slice(batch_start, batch_end - batch_start)
        
        batch_texts = batch["text"].to_list()
        batch_ids = batch["id"].to_list()
        batch_domains = batch["domain"].to_list()
        batch_dates = batch["date"].to_list()
        
        batch_count = 0
        for i in range(len(batch_texts)):
            speakers = extract_speakers(batch_texts[i])
            for speaker in speakers:
                fout.write(json.dumps({
                    "article_id": batch_ids[i],
                    "domain": batch_domains[i],
                    "date": batch_dates[i],
                    "speaker": speaker,
                }, ensure_ascii=False) + "\n")
                batch_count += 1
        
        total_quotes += batch_count
        del batch, batch_texts, batch_ids, batch_domains, batch_dates
        print(f"  Batch {batch_start//BATCH_SIZE + 1}: processed {batch_end:,} / {n_articles:,} articles, {batch_count:,} quotes in batch")

free_memory()
print(f"\nTotal: {total_quotes:,} speaker mentions written to {QUOTES_FILE}")

In [ ]:
# Drop text column — no longer needed, frees ~7GB
articles = articles.drop("text")
free_memory()

# Load quotes from JSONL (lightweight: just names, ids, dates)
quotes_df = pl.read_ndjson(QUOTES_FILE)
print(f"Quotes DataFrame: {len(quotes_df):,} rows")
print(f"Articles RAM after dropping text: {articles.estimated_size('mb'):.0f} MB")
quotes_df.head(10)

## 4. Inferência de género a partir do primeiro nome

Usamos uma abordagem em camadas:
1. Dicionário de nomes portugueses (INE / lista curada)
2. Biblioteca `gender-guesser` como fallback
3. Regras morfológicas (-a = F, -o = M) como último recurso

In [ ]:
import gender_guesser.detector as gender_detector

detector = gender_detector.Detector()

# Curated Portuguese names that gender-guesser may miss or get wrong
PT_NAMES_OVERRIDE = {
    # Female names common in Portugal
    "Conceição": "F", "Fátima": "F", "Graça": "F", "Florbela": "F",
    "Assunção": "F", "Encarnação": "F", "Piedade": "F", "Rosário": "F",
    "Guadalupe": "F", "Mafalda": "F", "Inês": "F", "Leonor": "F",
    "Catarina": "F", "Margarida": "F", "Constança": "F", "Filipa": "F",
    "Joana": "F", "Raquel": "F", "Beatriz": "F", "Matilde": "F",
    "Soraia": "F", "Cláudia": "F", "Sílvia": "F", "Mónica": "F",
    "Dulce": "F", "Lucinda": "F", "Celeste": "F", "Odete": "F",
    "Idalina": "F", "Elvira": "F", "Deolinda": "F", "Cremilde": "F",
    "Arlete": "F", "Isilda": "F", "Olinda": "F", "Noémia": "F",
    # Male names common in Portugal
    "Gonçalo": "M", "Vasco": "M", "Duarte": "M", "Henrique": "M",
    "Nuno": "M", "Rui": "M", "Tiago": "M", "Diogo": "M",
    "Afonso": "M", "Lourenço": "M", "Álvaro": "M", "Hélder": "M",
    "Joaquim": "M", "Jaime": "M", "Amílcar": "M", "Aníbal": "M",
    "Hermínio": "M", "Firmino": "M", "Arnaldo": "M", "Acácio": "M",
    # Ambiguous / tricky
    "Cruz": "U", "Paz": "U", "Sacramento": "U",
}

# Common titles/words that are NOT names (false positives from regex)
NOT_NAMES = {
    "O", "A", "Os", "As", "Um", "Uma", "Uns", "Umas",
    "Este", "Esta", "Esse", "Essa", "Aquele", "Aquela",
    "No", "Na", "Do", "Da", "De", "Em", "Por", "Para", "Com",
    "Que", "Se", "Já", "Mais", "Muito", "Também", "Ainda",
    "Governo", "Estado", "Portugal", "Lisboa", "Porto", "Europa",
    "Câmara", "Assembleia", "Parlamento", "Tribunal", "Ministério",
    "Presidente", "Primeiro", "Ministro", "Secretário", "Director",
    "Norte", "Sul", "Leste", "Oeste", "Centro",
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
    "Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo",
}


def infer_gender(full_name: str) -> str:
    """Infer gender from a Portuguese name. Returns 'M', 'F', or 'U' (unknown)."""
    if not full_name or not full_name.strip():
        return "U"
    
    first_name = full_name.strip().split()[0]
    
    # Filter out non-names
    if first_name in NOT_NAMES:
        return "X"  # X = not a name, should be filtered out
    
    # Check override dict first
    if first_name in PT_NAMES_OVERRIDE:
        return PT_NAMES_OVERRIDE[first_name]
    
    # Try gender-guesser
    result = detector.get_gender(first_name, "portugal")
    if result == "female" or result == "mostly_female":
        return "F"
    elif result == "male" or result == "mostly_male":
        return "M"
    
    # Try without country
    result = detector.get_gender(first_name)
    if result == "female" or result == "mostly_female":
        return "F"
    elif result == "male" or result == "mostly_male":
        return "M"
    
    # Morphological fallback for Portuguese
    if first_name.endswith("a") and not first_name.endswith("ista"):
        return "F"
    elif first_name.endswith("o") or first_name.endswith("os"):
        return "M"
    
    return "U"


# Test
test_names = ["António Costa", "Maria Silva", "Ana Gomes", "Rui Rio", "Catarina Martins", "Governo Regional"]
for name in test_names:
    print(f"  {name:25s} -> {infer_gender(name)}")

## 5. Aplicar inferência de género a todas as citações

In [ ]:
# Extract first name and infer gender
speakers_unique = quotes_df["speaker"].unique().to_list()
print(f"Unique speaker names: {len(speakers_unique):,}")

# Build gender lookup for unique names
gender_map = {name: infer_gender(name) for name in speakers_unique}

# Distribution check
gender_counts = Counter(gender_map.values())
print(f"\nGender distribution of unique names:")
for g, c in sorted(gender_counts.items()):
    label = {"M": "Male", "F": "Female", "U": "Unknown", "X": "Not a name"}.get(g, g)
    print(f"  {label:12s}: {c:>8,} ({100*c/len(speakers_unique):.1f}%)")

In [ ]:
# Add gender column to quotes DataFrame
quotes_df = quotes_df.with_columns(
    pl.col("speaker").replace(gender_map).alias("gender")
)

# Filter out non-names (X) and keep only valid gender assignments
quotes_valid = quotes_df.filter(pl.col("gender") != "X")
print(f"Valid quotes (excluding non-names): {len(quotes_valid):,} / {len(quotes_df):,}")
print(f"Filtered out {len(quotes_df) - len(quotes_valid):,} false positives")

quotes_valid.head(10)

## 6. Análise e estatísticas

In [ ]:
# Overall gender split (excluding unknowns)
quotes_known = quotes_valid.filter(pl.col("gender") != "U")

overall = quotes_known.group_by("gender").agg(pl.len().alias("count"))
total_known = overall["count"].sum()

print("=" * 50)
print("OVERALL GENDER SPLIT IN NEWS QUOTES")
print("=" * 50)
for row in overall.sort("gender").iter_rows(named=True):
    pct = 100 * row["count"] / total_known
    label = "Male" if row["gender"] == "M" else "Female"
    print(f"  {label:8s}: {row['count']:>10,} ({pct:.1f}%)")
print(f"  {'Total':8s}: {total_known:>10,}")
print(f"\n  Unknown gender: {len(quotes_valid) - len(quotes_known):,} quotes excluded")

In [ ]:
# Gender split by year
quotes_with_year = quotes_known.with_columns(
    pl.col("date").str.slice(0, 4).alias("year")
).filter(pl.col("year").str.len_chars() == 4)

by_year = (
    quotes_with_year
    .group_by(["year", "gender"])
    .agg(pl.len().alias("count"))
    .sort(["year", "gender"])
)

# Pivot for readability
by_year_pivot = by_year.pivot(on="gender", index="year", values="count").sort("year")
by_year_pivot = by_year_pivot.with_columns(
    (100 * pl.col("F") / (pl.col("F") + pl.col("M"))).round(1).alias("% Female")
)

print("Gender split by year:")
print(by_year_pivot)

In [ ]:
# Gender split by domain (top 30 domains by quote volume)
by_domain = (
    quotes_known
    .group_by(["domain", "gender"])
    .agg(pl.len().alias("count"))
)

domain_totals = by_domain.group_by("domain").agg(pl.col("count").sum().alias("total"))
top_domains = domain_totals.sort("total", descending=True).head(30)["domain"].to_list()

by_domain_top = by_domain.filter(pl.col("domain").is_in(top_domains))
by_domain_pivot = (
    by_domain_top
    .pivot(on="gender", index="domain", values="count")
    .with_columns(
        (pl.col("F").fill_null(0) + pl.col("M").fill_null(0)).alias("total"),
        (100 * pl.col("F").fill_null(0) / (pl.col("F").fill_null(0) + pl.col("M").fill_null(0))).round(1).alias("% Female")
    )
    .sort("total", descending=True)
)

print("Gender split by domain (top 30):")
print(by_domain_pivot.select(["domain", "M", "F", "total", "% Female"]))

In [ ]:
# Most quoted people overall
top_speakers = (
    quotes_known
    .group_by(["speaker", "gender"])
    .agg(pl.len().alias("mentions"))
    .sort("mentions", descending=True)
    .head(50)
)

print("Top 50 most quoted people:")
for row in top_speakers.iter_rows(named=True):
    g = "♂" if row["gender"] == "M" else "♀"
    print(f"  {g} {row['speaker']:30s} {row['mentions']:>6,} mentions")

## 6b. Estatísticas excluindo os mais citados

Os políticos/figuras públicas mais citados enviesam os dados — um único primeiro-ministro pode ter milhares de citações. Aqui vemos o panorama sem os top-N oradores.

In [ ]:
# Count mentions per speaker
speaker_counts = (
    quotes_known
    .group_by("speaker")
    .agg(pl.len().alias("mentions"))
    .sort("mentions", descending=True)
)

# Show thresholds
for n in [10, 25, 50, 100, 200]:
    top_n = set(speaker_counts.head(n)["speaker"].to_list())
    filtered = quotes_known.filter(~pl.col("speaker").is_in(top_n))
    total = len(filtered)
    n_f = filtered.filter(pl.col("gender") == "F").height
    n_m = filtered.filter(pl.col("gender") == "M").height
    pct_f = 100 * n_f / (n_f + n_m) if (n_f + n_m) > 0 else 0
    removed = len(quotes_known) - total
    print(f"Excluding top {n:>3d} speakers: {pct_f:5.1f}% female | {total:>10,} quotes remain ({removed:,} removed)")

In [ ]:
# Detailed breakdown excluding top 50 most quoted people
EXCLUDE_TOP_N = 50
top_n_speakers = set(speaker_counts.head(EXCLUDE_TOP_N)["speaker"].to_list())

quotes_excl = quotes_known.filter(~pl.col("speaker").is_in(top_n_speakers))
print(f"Excluding top {EXCLUDE_TOP_N} speakers ({len(quotes_known) - len(quotes_excl):,} quotes removed)\n")

# Overall
n_f = quotes_excl.filter(pl.col("gender") == "F").height
n_m = quotes_excl.filter(pl.col("gender") == "M").height
print(f"Overall (excl. top {EXCLUDE_TOP_N}):")
print(f"  Female: {n_f:>10,} ({100*n_f/(n_f+n_m):.1f}%)")
print(f"  Male:   {n_m:>10,} ({100*n_m/(n_f+n_m):.1f}%)")

# By year
excl_with_year = quotes_excl.with_columns(
    pl.col("date").str.slice(0, 4).alias("year")
).filter(pl.col("year").str.len_chars() == 4)

by_year_excl = (
    excl_with_year
    .group_by(["year", "gender"])
    .agg(pl.len().alias("count"))
    .pivot(on="gender", index="year", values="count")
    .sort("year")
    .with_columns(
        (100 * pl.col("F").fill_null(0) / (pl.col("F").fill_null(0) + pl.col("M").fill_null(0))).round(1).alias("% Female")
    )
)

print(f"\nBy year (excl. top {EXCLUDE_TOP_N}):")
print(by_year_excl)

# Who was excluded?
print(f"\nExcluded speakers:")
for row in speaker_counts.head(EXCLUDE_TOP_N).iter_rows(named=True):
    g = quotes_known.filter(pl.col("speaker") == row["speaker"]).select("gender").row(0)[0]
    g_icon = "♂" if g == "M" else "♀"
    print(f"  {g_icon} {row['speaker']:30s} {row['mentions']:>6,} mentions")

In [ ]:
import matplotlib.pyplot as plt

def yearly_gender_share(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.with_columns(pl.col("date").str.slice(0, 4).alias("year"))
        .filter(pl.col("year").str.len_chars() == 4)
        .group_by(["year", "gender"])
        .agg(pl.len().alias("count"))
        .pivot(on="gender", index="year", values="count")
        .with_columns(
            pl.col("F").fill_null(0).alias("F"),
            pl.col("M").fill_null(0).alias("M"),
        )
        .with_columns(
            (pl.col("F") + pl.col("M")).alias("total"),
            (100 * pl.col("F") / (pl.col("F") + pl.col("M"))).alias("pct_female"),
            (100 * pl.col("M") / (pl.col("F") + pl.col("M"))).alias("pct_male"),
        )
        .filter(pl.col("total") > 0)
        .select(["year", "pct_female", "pct_male"])
        .sort("year")
    )

# 1) Everyone
year_all = yearly_gender_share(quotes_known).rename({
    "pct_female": "All Female",
    "pct_male": "All Male",
})

# 2) Excluding speakers with > 100 mentions
over_100 = speaker_counts.filter(pl.col("mentions") > 100)["speaker"].to_list()
quotes_no_top100 = quotes_known.filter(~pl.col("speaker").is_in(over_100))
year_no_top100 = yearly_gender_share(quotes_no_top100).rename({
    "pct_female": "Excl>100 Female",
    "pct_male": "Excl>100 Male",
})

# Join for plotting
plot_df = (
    year_all.join(year_no_top100, on="year", how="outer")
    .with_columns(pl.coalesce(["year", "year_right"]).alias("year"))
    .drop("year_right")
    .sort("year")
)

x = [int(y) for y in plot_df["year"].to_list()]

plt.figure(figsize=(12, 6))
plt.plot(x, plot_df["All Female"].to_list(), marker="o", linewidth=2, label="All Female")
plt.plot(x, plot_df["All Male"].to_list(), marker="o", linewidth=2, label="All Male")
plt.plot(x, plot_df["Excl>100 Female"].to_list(), marker="o", linewidth=2, linestyle="--", label="Excl>100 Female")
plt.plot(x, plot_df["Excl>100 Male"].to_list(), marker="o", linewidth=2, linestyle="--", label="Excl>100 Male")

plt.title("Evolution of Female/Male Share in Quotes Over Time")
plt.xlabel("Year")
plt.ylabel("Share (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Excluded speakers with >100 mentions: {len(over_100)}")
print(f"Quotes kept after exclusion: {len(quotes_no_top100):,} / {len(quotes_known):,}")

## 7. Exportar resultados

In [ ]:
# OUTPUT_DIR = Path("output")
# OUTPUT_DIR.mkdir(exist_ok=True)

# # Export: article_id, speaker, gender as JSONL
# output_file = OUTPUT_DIR / "quotes_with_gender.jsonl"
# quotes_valid.select(["article_id", "speaker", "gender"]).write_ndjson(output_file)

# print(f"Exported {len(quotes_valid):,} rows to {output_file}")

## 8. Validação manual (amostra)

Verificar qualidade da extracção numa amostra aleatória.

In [ ]:
# Show random sample of extractions for manual validation
sample = quotes_valid.sample(20, seed=412)

for row in sample.iter_rows(named=True):
    g_icon = {"M": "♂", "F": "♀", "U": "?"}.get(row["gender"], "?")
    print(f"{g_icon} [{row['gender']}] {row['speaker']:30s} | {row['domain']:30s} | {row['article_id']}")
    print()

## 9. Comparação com dados oficiais IRN (Instituto dos Registos e do Notariado)

O dataset `pt_first_names.csv` contém ~103k nomes próprios portugueses com classificação M/F, obtido do registo civil. Vamos usar esta lista como fonte primária e comparar com os resultados anteriores (gender-guesser + heurísticas).

In [ ]:
# Load IRN official name-gender dataset
import csv
from collections import defaultdict

irn_raw = defaultdict(lambda: {"M": 0, "F": 0})

with open("data/pt_first_names.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row["name"].strip().upper()
        gender = row["classification"].strip()
        if gender in ("M", "F"):
            irn_raw[name][gender] += 1

# Resolve conflicts by majority (e.g. HELDER: M=3, F=1 → M)
irn_names = {}
for name, counts in irn_raw.items():
    if counts["M"] > counts["F"]:
        irn_names[name] = "M"
    elif counts["F"] > counts["M"]:
        irn_names[name] = "F"
    else:
        irn_names[name] = "U"  # tie = unknown

print(f"IRN dataset: {len(irn_names):,} unique names")
print(f"  Male: {sum(1 for v in irn_names.values() if v == 'M'):,}")
print(f"  Female: {sum(1 for v in irn_names.values() if v == 'F'):,}")
print(f"  Ambiguous: {sum(1 for v in irn_names.values() if v == 'U'):,}")

In [ ]:
import unicodedata

def normalize_name(name: str) -> str:
    """Normalize name for lookup: uppercase, strip accents."""
    upper = name.upper()
    # Also create accent-stripped version for fallback
    return upper

def strip_accents(s: str) -> str:
    """Remove diacritics from a string."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def infer_gender_irn(full_name: str) -> str:
    """Infer gender using IRN official data as primary source."""
    if not full_name or not full_name.strip():
        return "U"
    
    first_name = full_name.strip().split()[0]
    
    # Filter out non-names
    if first_name in NOT_NAMES:
        return "X"
    
    first_upper = first_name.upper()
    
    # 1. Direct lookup in IRN data
    if first_upper in irn_names:
        return irn_names[first_upper]
    
    # 2. Try without accents (IRN has both FATIMA and FÁTIMA)
    first_no_accents = strip_accents(first_upper)
    if first_no_accents in irn_names:
        return irn_names[first_no_accents]
    
    # 3. Try with common accent patterns
    # Some names in articles have accents but IRN doesn't, or vice versa
    for name_in_db, gender in irn_names.items():
        if strip_accents(name_in_db) == first_no_accents:
            return gender
            break  # just take the first match
    
    # 4. Fallback to gender-guesser
    result = detector.get_gender(first_name, "portugal")
    if result in ("female", "mostly_female"):
        return "F"
    elif result in ("male", "mostly_male"):
        return "M"
    
    result = detector.get_gender(first_name)
    if result in ("female", "mostly_female"):
        return "F"
    elif result in ("male", "mostly_male"):
        return "M"
    
    # 5. Morphological fallback
    if first_name.endswith("a") and not first_name.endswith("ista"):
        return "F"
    elif first_name.endswith("o") or first_name.endswith("os"):
        return "M"
    
    return "U"


# Test
test_names = ["António Costa", "Maria Silva", "Ana Gomes", "Rui Rio", "Catarina Martins", 
              "Hélder Rodrigues", "Conceição Lopes", "Governo Regional"]
print("IRN-based gender inference:")
for name in test_names:
    print(f"  {name:25s} -> {infer_gender_irn(name)}")

In [ ]:
# Rebuild gender map using IRN-based classifier
gender_map_irn = {name: infer_gender_irn(name) for name in speakers_unique}

# Compare with previous method
irn_counts = Counter(gender_map_irn.values())
old_counts = Counter(gender_map.values())

print(f"{'Method':<20s} {'M':>8s} {'F':>8s} {'U':>8s} {'X':>8s}")
print("-" * 50)
print(f"{'Old (gender-guesser)':<20s} {old_counts.get('M',0):>8,} {old_counts.get('F',0):>8,} {old_counts.get('U',0):>8,} {old_counts.get('X',0):>8,}")
print(f"{'New (IRN + fallback)':<20s} {irn_counts.get('M',0):>8,} {irn_counts.get('F',0):>8,} {irn_counts.get('U',0):>8,} {irn_counts.get('X',0):>8,}")

# How many changed?
changed = sum(1 for name in speakers_unique if gender_map[name] != gender_map_irn[name])
print(f"\nNames that changed classification: {changed:,} / {len(speakers_unique):,} ({100*changed/len(speakers_unique):.1f}%)")

# What types of changes?
changes = Counter()
for name in speakers_unique:
    old = gender_map[name]
    new = gender_map_irn[name]
    if old != new:
        changes[f"{old}->{new}"] += 1

print("\nChange breakdown:")
for change, count in changes.most_common():
    print(f"  {change}: {count:,}")

In [ ]:
# Apply IRN-based gender and compare overall stats
quotes_df_irn = quotes_df.with_columns(
    pl.col("speaker").replace(gender_map_irn).alias("gender_irn")
)

quotes_valid_irn = quotes_df_irn.filter(pl.col("gender_irn") != "X")
quotes_known_irn = quotes_valid_irn.filter(pl.col("gender_irn") != "U")

# Compare overall splits
old_f = quotes_known.filter(pl.col("gender") == "F").height
old_m = quotes_known.filter(pl.col("gender") == "M").height
new_f = quotes_known_irn.filter(pl.col("gender_irn") == "F").height
new_m = quotes_known_irn.filter(pl.col("gender_irn") == "M").height

print("=" * 60)
print("COMPARISON: Old method vs IRN-based")
print("=" * 60)
print(f"{'':20s} {'Old method':>15s} {'IRN-based':>15s}")
print(f"{'Female quotes':20s} {old_f:>12,}   {new_f:>12,}")
print(f"{'Male quotes':20s} {old_m:>12,}   {new_m:>12,}")
print(f"{'Total known':20s} {old_f+old_m:>12,}   {new_f+new_m:>12,}")
print(f"{'% Female':20s} {100*old_f/(old_f+old_m):>11.1f}%   {100*new_f/(new_f+new_m):>11.1f}%")
print(f"{'Unknown':20s} {len(quotes_valid)-len(quotes_known):>12,}   {len(quotes_valid_irn)-len(quotes_known_irn):>12,}")

# By year comparison
quotes_year_irn = quotes_known_irn.with_columns(
    pl.col("date").str.slice(0, 4).alias("year")
).filter(pl.col("year").str.len_chars() == 4)

by_year_irn = (
    quotes_year_irn
    .group_by(["year", "gender_irn"])
    .agg(pl.len().alias("count"))
    .pivot(on="gender_irn", index="year", values="count")
    .sort("year")
    .with_columns(
        (100 * pl.col("F").fill_null(0) / (pl.col("F").fill_null(0) + pl.col("M").fill_null(0))).round(1).alias("% Female (IRN)")
    )
)

print("\n\nBy year (IRN-based):")
print(by_year_irn)

In [ ]:
# exportar versão IRN
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Export IRN-based results: article_id, speaker, gender as JSONL
output_file = OUTPUT_DIR / "quotes_with_gender.jsonl"
quotes_valid_irn.select(["article_id", "speaker", pl.col("gender_irn").alias("gender")]).write_ndjson(output_file)

print(f"Exported {len(quotes_valid_irn):,} rows to {output_file} (IRN-based gender)")